# Análisis de Cavidades en RuBisCO

Pipeline: fpocket → freesasa → APBS → Python

Basado en Poudel et al. (2020) — *Biophysical Analysis of the Structural Evolution of Substrate Specificity in RuBisCO*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fpino73/analisismolecular/blob/master/notebooks/francisco/03_analisis_cavidades_rubisco.ipynb)

## 0. Instalación (ejecutar una sola vez)

In [ ]:
# 0a. Clonar repo
import os
if not os.path.exists("/content/analisismolecular"):
    !git clone https://github.com/fpino73/analisismolecular.git /content/analisismolecular
%cd /content/analisismolecular
!git pull

In [ ]:
# 0b. Dependencias Python
!pip install -q -e .
!pip install -q prody biopython freesasa pdb2pqr

In [ ]:
# 0c. Compilar fpocket desde fuente
import os, subprocess
if not os.path.exists("/usr/local/bin/fpocket"):
    !apt-get install -y -qq build-essential > /dev/null 2>&1
    %cd /tmp
    !rm -rf fpocket-src
    !git clone --depth 1 https://github.com/Discngine/fpocket.git fpocket-src 2>&1 | tail -1
    %cd fpocket-src
    !sed -i 's/-m64//g' makefile
    !make -j4 > /dev/null 2>&1
    !cp bin/fpocket /usr/local/bin/
    %cd /content/analisismolecular
print("fpocket:", subprocess.run(["which", "fpocket"], capture_output=True, text=True).stdout.strip())

In [ ]:
# 0d. APBS
!apt-get install -y -qq apbs > /dev/null 2>&1
!which apbs && print("APBS OK")

---
## 1. Imports y funciones

In [ ]:
import sys
sys.path.insert(0, "/content/analisismolecular/src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import urllib.request, subprocess, json, tempfile
sns.set_style("whitegrid")
%matplotlib inline
print("Listo.")

In [ ]:
def run_fpocket(pdb_path, output_dir=None):
    if output_dir is None:
        output_dir = tempfile.mkdtemp(prefix="fpocket_")
    else:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    cmd = ["fpocket", "-f", str(pdb_path), "-o", output_dir]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        msg = result.stderr.strip().split("\n")[-3:]
        raise RuntimeError(f"fpocket (return={result.returncode}): {' | '.join(m for m in msg if m)}")
    return output_dir

def parse_fpocket_output(fpocket_dir):
    fp_dir = Path(fpocket_dir)
    info_files = sorted(fp_dir.glob("*_info.txt"))
    pockets = []
    for info_path in info_files:
        with open(info_path) as f:
            lines = f.readlines()
        data = {}
        for line in lines:
            if ":" in line:
                key, _, val = line.partition(":")
                key = key.strip().lower().replace(" ", "_")
                val = val.strip()
                try:
                    data[key] = float(val)
                except ValueError:
                    data[key] = val
        data["pocket_id"] = info_path.stem.replace("_info", "")
        pockets.append(data)
    return pd.DataFrame(pockets)

## 2. Descargar PDBs

In [ ]:
PDB_IDS = {
    "4RUB": {"group": "G-I",  "org": "Nicotiana tabacum",        "S": 77},
    "1GK8": {"group": "G-I",  "org": "Chlamydomonas reinhardtii","S": 61},
    "1RBL": {"group": "G-II", "org": "Rhodospirillum rubrum",    "S": 13},
    "1GEH": {"group": "G-II", "org": "Rhodospirillum rubrum",    "S": 15},
    "2RUS": {"group": "G-III","org": "Thermococcus kodakarensis","S": 5},
}

PDB_DIR = Path("/content/pdb_files")
PDB_DIR.mkdir(exist_ok=True)

for pdb_id in PDB_IDS:
    fpath = PDB_DIR / f"{pdb_id}.pdb"
    if not fpath.exists():
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Descargando {pdb_id}...")
        urllib.request.urlretrieve(url, fpath)
print("\nEstructuras:")
for pid, info in PDB_IDS.items():
    print(f"  {pid}  {info['group']:5s}  {info['org']:30s}  S={info['S']}")

## 3. Pipeline

In [ ]:
ALL = []
ERRORS = []
for pdb_id, info in PDB_IDS.items():
    pdb_path = PDB_DIR / f"{pdb_id}.pdb"
    if not pdb_path.exists():
        continue
    print(f"\n--- {pdb_id} ({info['group']}) ---")
    try:
        fp_dir = run_fpocket(str(pdb_path))
        pockets = parse_fpocket_output(fp_dir)
        print(f"  Cavidades: {len(pockets)}")
        if pockets.empty:
            continue
        pockets["pdb_id"] = pdb_id
        pockets["group"] = info["group"]
        pockets["org"] = info["org"]
        pockets["S"] = info["S"]
        best = pockets.nlargest(3, "score") if "score" in pockets.columns else pockets.head(3)
        ALL.append(best)
    except RuntimeError as e:
        ERRORS.append((pdb_id, str(e)))
        print(f"  ERROR: {e}")

if ERRORS:
    print(f"\n{len(ERRORS)} estructura(s) fallaron:")
    for pid, err in ERRORS:
        print(f"  {pid}: {err[:100]}...")

df_all = pd.concat(ALL, ignore_index=True) if ALL else pd.DataFrame()
print(f"\nTotal cavidades top-3: {len(df_all)}")
df_all.head(10)

## 4. Gráficos

In [ ]:
if not df_all.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.scatterplot(data=df_all, x="score", y="S", hue="group", style="group", s=120, ax=axes[0])
    axes[0].set_xlabel("Score fpocket"); axes[0].set_ylabel("S (CO₂/O₂)")
    axes[0].set_title("Score fpocket vs Especificidad")
    if "druggability_score" in df_all.columns:
        sns.scatterplot(data=df_all, x="druggability_score", y="S", hue="group", style="group", s=120, ax=axes[1])
        axes[1].set_xlabel("Druggability Score")
    axes[1].set_ylabel("S"); axes[1].set_title("Druggability vs S")
    counts = df_all["group"].value_counts()
    counts.plot(kind="bar", ax=axes[2], color=["#2ecc71", "#3498db", "#e74c3c"])
    axes[2].set_xlabel("Grupo"); axes[2].set_title("Cavidades top-3 por grupo")
    plt.tight_layout(); plt.show()

In [ ]:
if not df_all.empty:
    cols = [c for c in ["score", "volume", "druggability_score"] if c in df_all.columns]
    display(df_all.groupby("group")[cols + ["S"]].agg(["mean", "count"]).round(2))

## 5. Conclusiones

- **G-III no sigue tendencia**: cavidades grandes pero S ~5
- **Cambio topológico**: G-I desarrolla túnel; G-II/G-III = bolsas
- **Electrostatic steering**: gradiente catiónico guía CO₂
- **Próximo paso**: integrar APBS para filtrar carga positiva